# Reliability scoring for the BankBench-MY Tamper ScorecardApplies **Dimension 3.5 (Reliability)** of the AISL Scorecard to this folder's eval. Expected to surface the N=1 single-pass gap (mirror of the standard_scorecard roadmap's expectation for this dimension).

In [ ]:
# Ported verbatim from bankbench/standard_scorecard/01_construct_validity.ipynb# (implements the AISL paper's Table 1 aggregation rule, Sec 3.1).SEVERITY_SCORE = {"yellow": 2, "orange": 3, "red": 4}def score_dimension(items):    """items: list of dicts with keys principle, subitem, applies_to,    highlight, satisfied, notes. satisfied: True / False / None    (None = not_applicable). Returns (score, verdict_str)."""    applicable = [i for i in items if i["satisfied"] is not None]    unsatisfied = [i for i in applicable if i["satisfied"] is False]    non_highlighted_unsatisfied = [i for i in unsatisfied if i["highlight"] is None]    if non_highlighted_unsatisfied:        names = "; ".join(i["subitem"] for i in non_highlighted_unsatisfied)        return None, f"INVALID - non-highlighted item(s) unsatisfied: {names}"    highlighted_unsatisfied = [i for i in unsatisfied if i["highlight"] is not None]    if not highlighted_unsatisfied:        return 1, "Score 1 - every applicable item satisfied"    worst = max(highlighted_unsatisfied, key=lambda i: SEVERITY_SCORE[i["highlight"]])    score = SEVERITY_SCORE[worst["highlight"]]    return score, f"Score {score} - downgraded by: {worst['subitem']} ({worst['highlight']})"

In [ ]:
reliability_items = [    {        "principle": "Replicability",        "subitem": "Disclose model versions, parameters, and infrastructure",        "applies_to": "all",        "highlight": null,        "satisfied": true,        "notes": "MODELS (id/label/provider/base_url/key_env) are exported in tamper_eval_results_live.json; temperature=0, max_tokens per model documented."    },    {        "principle": "Replicability",        "subitem": "Log seeds and random state",        "applies_to": "all",        "highlight": null,        "satisfied": false,        "notes": "temperature=0 is set, but no explicit seed is logged per run/cell - API determinism is trusted, not verified."    },    {        "principle": "Replicability",        "subitem": "Multiple runs to bound variance (N>=3)",        "applies_to": "all",        "highlight": "orange",        "satisfied": false,        "notes": "RUNS_PER_CELL defaults to 1 (pilot); beta B upgrade requires RUNS_PER_CELL>=3, matching the reference's pre-registration note."    },    {        "principle": "Reproducibility",        "subitem": "Version-lock dependencies",        "applies_to": "all",        "highlight": null,        "satisfied": false,        "notes": "requirements.txt pins packages but not exact versions; no lockfile for the Python eval side (sandbox has package-lock.json)."    },    {        "principle": "Reproducibility",        "subitem": "Make raw outputs available",        "applies_to": "all",        "highlight": null,        "satisfied": true,        "notes": "Per-cell raw model responses are kept in the exported JSON (runs[].raw) and shown in the dashboard drill-down."    }]

In [ ]:
score, verdict = score_dimension(reliability_items)print(verdict)for i in reliability_items:    if i["satisfied"] is False:        tag = f"[{i['highlight']}]" if i["highlight"] else "[unhighlighted]"        print(f"  {tag:12s} {i['subitem']}")

In [ ]:
import json, datetimeresult = {"dimension": "Reliability", "scored_at": datetime.date.today().isoformat(), "score": score, "verdict": verdict, "items": reliability_items}with open("results/reliability.json", "w") as f:    json.dump(result, f, indent=2, default=str)print("Wrote results/reliability.json")